<a href="https://colab.research.google.com/github/robertkigobe/llm_engineering/blob/main/Fina_log_analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import userdata
import os
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

In [ ]:
!pip -q install pandas==2.2.2 gradio anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 8.9 MB/s eta 0:00:00


In [ ]:
# =============================================================================
# Log Analysis Agent — Claude + Gradio
# Run each cell in order in Google Colab
# =============================================================================


# ── Cell 1: Install ───────────────────────────────────────────────────────────
# !pip -q install pandas==2.2.2 gradio anthropic


# ── Cell 2: Imports ───────────────────────────────────────────────────────────
import pandas as pd
import gradio as gr
import anthropic

# ── Cell 3: Global state ──────────────────────────────────────────────────────
uploaded_df = None


# ── Cell 4: Schema normalisation ─────────────────────────────────────────────
def normalize_schema(df: pd.DataFrame) -> pd.DataFrame:
    """Map vendor-specific column names to canonical names."""
    COLUMN_ALIASES = {
        # log level
        "level":        "log_level",
        "severity":     "log_level",
        "loglevel":     "log_level",
        # error type
        "errorType":    "error_type",
        "exception":    "error_type",
        "error":        "error_type",
        "err_type":     "error_type",
        # message
        "errorMessage": "message",
        "msg":          "message",
        "log_message":  "message",
        # service
        "app":          "service",
        "application":  "service",
        "service_name": "service",
    }
    df = df.copy()
    for src, dst in COLUMN_ALIASES.items():
        if src in df.columns and dst not in df.columns:
            df[dst] = df[src]
    if "log_level" in df.columns:
        df["log_level"] = df["log_level"].astype(str).str.upper()
    return df


# ── Cell 5: File loader ───────────────────────────────────────────────────────
def load_logs_gradio(file):
    """Load CSV / JSONL / XLSX, normalise schema, store globally."""
    global uploaded_df

    if file is None:
        uploaded_df = None
        return pd.DataFrame({"message": ["No file uploaded yet."]})

    # Gradio 4+ gives a plain string path; older versions give a file object
    path = file if isinstance(file, str) else file.name

    if path.endswith(".csv"):
        df = pd.read_csv(path)
    elif path.endswith(".jsonl") or path.endswith(".ndjson"):
        df = pd.read_json(path, lines=True)
    elif path.endswith(".xlsx"):
        df = pd.read_excel(path)
    else:
        uploaded_df = None
        return pd.DataFrame({"error": [f"Unsupported file type: {path}"]})

    # Parse timestamps
    if "@timestamp" in df.columns:
        df["@timestamp"] = pd.to_datetime(df["@timestamp"], errors="coerce", utc=True)
    elif "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)

    uploaded_df = normalize_schema(df)
    return uploaded_df.head(15)


# ── Cell 6: Dataset profile ───────────────────────────────────────────────────
def dataset_profile() -> str:
    """Return a plain-text summary of the loaded dataset."""
    global uploaded_df
    if uploaded_df is None or uploaded_df.empty:
        return "No dataset loaded. Please upload a log file first."

    lines = [
        f"Rows: {len(uploaded_df):,}   Columns: {len(uploaded_df.columns)}",
        f"Column names: {', '.join(uploaded_df.columns.tolist())}",
    ]
    if "log_level" in uploaded_df.columns:
        lines.append(f"Log levels: {uploaded_df['log_level'].value_counts().to_dict()}")
    if "service" in uploaded_df.columns:
        services = sorted(uploaded_df["service"].dropna().unique().tolist())
        lines.append(f"Services ({len(services)}): {', '.join(services)}")
    if "error_type" in uploaded_df.columns:
        etypes = sorted(uploaded_df["error_type"].dropna().unique().tolist())
        lines.append(f"Error types ({len(etypes)}): {', '.join(etypes)}")

    ts_col = (
        "@timestamp" if "@timestamp" in uploaded_df.columns
        else "timestamp" if "timestamp" in uploaded_df.columns
        else None
    )
    if ts_col:
        ts = pd.to_datetime(uploaded_df[ts_col], errors="coerce", utc=True)
        lines.append(f"Date range: {ts.min().date()} → {ts.max().date()}")

    return "\n".join(lines)


# ── Cell 7: Query helpers ─────────────────────────────────────────────────────
def ensure_data_loaded():
    if uploaded_df is None or uploaded_df.empty:
        return False, "No data loaded yet. Please upload a log file first."
    return True, None


def normalize_timestamp(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "@timestamp" in df.columns:
        df["_ts"] = pd.to_datetime(df["@timestamp"], errors="coerce", utc=True)
    elif "timestamp" in df.columns:
        df["_ts"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
    else:
        df["_ts"] = pd.NaT
    return df


def get_error_summary():
    ok, msg = ensure_data_loaded()
    if not ok:
        return msg
    df = normalize_timestamp(uploaded_df)
    result = (
        df[df["log_level"] == "ERROR"]
        .groupby("error_type")
        .size()
        .reset_index(name="error_count")
        .sort_values("error_count", ascending=False)
    )
    return result


def get_top_errors(limit: int = 5):
    ok, msg = ensure_data_loaded()
    if not ok:
        return msg
    result = (
        uploaded_df[uploaded_df["log_level"] == "ERROR"]
        .groupby(["error_type", "service"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .head(int(limit))
    )
    return result


def get_errors_by_service(service_name: str):
    ok, msg = ensure_data_loaded()
    if not ok:
        return msg
    df = normalize_timestamp(uploaded_df)
    filtered = df[
        (df["log_level"] == "ERROR") &
        (df["service"] == service_name)
    ]
    if filtered.empty:
        return f"No errors found for service: {service_name}"
    cols = [c for c in ["_ts", "service", "error_type", "message", "sourceId"]
            if c in filtered.columns]
    return (
        filtered[cols]
        .rename(columns={"_ts": "timestamp"})
        .sort_values("timestamp", ascending=False)
        .head(50)
    )


def get_errors_in_date_range(start_date: str, end_date: str):
    ok, msg = ensure_data_loaded()
    if not ok:
        return msg
    df = normalize_timestamp(uploaded_df)
    start_dt = pd.to_datetime(start_date, errors="coerce", utc=True)
    end_dt   = pd.to_datetime(end_date,   errors="coerce", utc=True)
    if pd.isna(start_dt) or pd.isna(end_dt):
        return "Invalid date format — use YYYY-MM-DD."
    mask = (
        (df["_ts"] >= start_dt) &
        (df["_ts"] <= end_dt) &
        (df["log_level"] == "ERROR")
    )
    result = df.loc[mask]
    if result.empty:
        return "No errors found in that date range."
    cols = [c for c in ["_ts", "service", "error_type", "message"]
            if c in result.columns]
    return (
        result[cols]
        .rename(columns={"_ts": "timestamp"})
        .sort_values("timestamp", ascending=False)
        .head(100)
    )


# ── Cell 8: Claude client + tool definitions ──────────────────────────────────
# Set ANTHROPIC_API_KEY in Colab:
#   from google.colab import userdata
#   import os; os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

client = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY from env
MODEL  = "claude-opus-4-6"

TOOLS = [
    {
        "name": "get_dataset_profile",
        "description": (
            "Return metadata about the loaded dataset — row count, column names, "
            "available services, error types, and date range. "
            "Always call this first when the user asks what data is available."
        ),
        "input_schema": {"type": "object", "properties": {}, "required": []},
    },
    {
        "name": "get_error_summary",
        "description": (
            "Return a frequency table of all error types across the dataset. "
            "Use for a broad overview of which errors occur most often."
        ),
        "input_schema": {"type": "object", "properties": {}, "required": []},
    },
    {
        "name": "get_top_errors",
        "description": (
            "Return the top-N most frequent (error_type, service) combinations. "
            "Use when the analyst wants to know the biggest problems."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "limit": {
                    "type": "integer",
                    "description": "Number of rows to return. Default 5.",
                }
            },
            "required": [],
        },
    },
    {
        "name": "get_errors_by_service",
        "description": (
            "Return recent errors for one specific service. "
            "Use when the analyst asks about a named service or application."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "service_name": {
                    "type": "string",
                    "description": "Exact name of the service to filter on.",
                }
            },
            "required": ["service_name"],
        },
    },
    {
        "name": "get_errors_in_date_range",
        "description": (
            "Return errors between two dates (inclusive). "
            "Use when the analyst mentions a specific time window."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "start_date": {
                    "type": "string",
                    "description": "Start date in YYYY-MM-DD format.",
                },
                "end_date": {
                    "type": "string",
                    "description": "End date in YYYY-MM-DD format.",
                },
            },
            "required": ["start_date", "end_date"],
        },
    },
]


# ── Cell 9: Tool dispatcher ───────────────────────────────────────────────────
import traceback
import time

def _execute_tool(name: str, inputs: dict) -> str:
    """Run the named tool and return a Markdown string for Claude to read."""
    print(f"\n[TOOL CALL] {name}  inputs={inputs}")
    try:
        if name == "get_dataset_profile":
            result = dataset_profile()
        elif name == "get_error_summary":
            result = get_error_summary()
        elif name == "get_top_errors":
            result = get_top_errors(int(inputs.get("limit", 5)))
        elif name == "get_errors_by_service":
            result = get_errors_by_service(inputs["service_name"])
        elif name == "get_errors_in_date_range":
            result = get_errors_in_date_range(inputs["start_date"], inputs["end_date"])
        else:
            print(f"[TOOL ERROR] Unknown tool: {name}")
            return f"Unknown tool: {name}"

        if isinstance(result, pd.DataFrame):
            out = "No results found." if result.empty else result.to_markdown(index=False)
        else:
            out = str(result)

        print(f"[TOOL RESULT] {name}:\n{out[:500]}{'...' if len(out) > 500 else ''}")
        return out

    except Exception as exc:
        tb = traceback.format_exc()
        print(f"[TOOL EXCEPTION] {name}:\n{tb}")
        return f"Tool error ({name}): {exc}"


# ── Cell 10: Agentic loop with streaming ─────────────────────────────────────
SYSTEM = """\
You are an expert log-analysis assistant helping non-technical analysts \
understand application logs.

Rules:
1. Always call get_dataset_profile first if you are unsure what data is loaded.
2. Call a tool to fetch real data before drawing any conclusions — never invent numbers.
3. Explain findings in plain English with no jargon.
4. Highlight the single most important pattern and suggest one concrete next step.
5. If no dataset is loaded, politely ask the user to upload a log file.\
"""

# Custom status messages shown while the agent is working
THINKING_MESSAGES = [
    "Biz apps demo working hard... 🔧",
    "CDS at work... 📊",
    "Thinking aloud... 🤔",
    "Avoiding mistakes... 🎯",
    "Connecting the dots... 🔗",
    "Reading the logs carefully... 🔍",
]


def run_agent(user_msg: str, history: list):
    """
    Streaming agentic loop. Yields intermediate status updates to the UI
    while Claude is working, then streams the final answer token by token.
    """
    if not user_msg.strip():
        yield "", history
        return

    print(f"\n{'='*60}")
    print(f"[USER] {user_msg}")
    print(f"{'='*60}")

    # Show first thinking message immediately
    thinking_idx = 0
    working_history = history + [[user_msg, f"_{THINKING_MESSAGES[thinking_idx]}_"]]
    yield "", working_history

    try:
        # Build messages from Gradio history
        messages = []
        for turn in history:
            human, assistant = turn[0], turn[1]
            messages.append({"role": "user",      "content": human})
            messages.append({"role": "assistant", "content": assistant})
        messages.append({"role": "user", "content": user_msg})

        final_text = ""
        turn_count = 0

        while True:
            turn_count += 1

            # Rotate thinking message on each API call
            thinking_idx = (thinking_idx + 1) % len(THINKING_MESSAGES)
            working_history = history + [[user_msg, f"_{THINKING_MESSAGES[thinking_idx]}_"]]
            yield "", working_history

            print(f"\n[API CALL #{turn_count}] Sending {len(messages)} messages to {MODEL}...")

            response = client.messages.create(
                model=MODEL,
                max_tokens=4096,
                system=SYSTEM,
                tools=TOOLS,
                messages=messages,
            )

            print(f"[API RESPONSE] stop_reason={response.stop_reason}  "
                  f"blocks={[b.type for b in response.content]}")

            # Claude is done — stream the final answer word by word
            if response.stop_reason == "end_turn":
                raw_text = "\n".join(
                    b.text for b in response.content if b.type == "text"
                )
                print(f"[FINAL ANSWER]\n{raw_text[:500]}")

                # Stream word by word into the chat
                streamed = ""
                for word in raw_text.split(" "):
                    streamed += word + " "
                    stream_history = history + [[user_msg, streamed]]
                    yield "", stream_history
                    time.sleep(0.03)   # ~33 words/sec — feels natural

                final_text = raw_text
                break

            # Claude wants to call tools
            if response.stop_reason == "tool_use":
                tool_uses = [b for b in response.content if b.type == "tool_use"]
                print(f"[TOOL USES] {[tu.name for tu in tool_uses]}")

                # Show which tool is being called
                tool_names = ", ".join(tu.name for tu in tool_uses)
                thinking_idx = (thinking_idx + 1) % len(THINKING_MESSAGES)
                status = (
                    f"_{THINKING_MESSAGES[thinking_idx]}_\n\n"
                    f"🛠️ Running: `{tool_names}`..."
                )
                working_history = history + [[user_msg, status]]
                yield "", working_history

                messages.append({"role": "assistant", "content": response.content})

                tool_results = [
                    {
                        "type":        "tool_result",
                        "tool_use_id": tu.id,
                        "content":     _execute_tool(tu.name, tu.input),
                    }
                    for tu in tool_uses
                ]
                messages.append({"role": "user", "content": tool_results})
                continue

            # Unexpected stop reason
            final_text = (
                "\n".join(b.text for b in response.content if b.type == "text")
                or f"Unexpected stop reason: {response.stop_reason}"
            )
            print(f"[UNEXPECTED STOP] {response.stop_reason}")
            break

        final_history = history + [[user_msg, final_text]]
        yield "", final_history

    except Exception as e:
        tb = traceback.format_exc()
        print(f"\n[RUN_AGENT EXCEPTION]\n{tb}")
        error_msg = (
            f"❌ **Error: {type(e).__name__}**\n\n"
            f"```\n{tb}\n```"
        )
        yield "", history + [[user_msg, error_msg]]


# ── Cell 11: Gradio UI ────────────────────────────────────────────────────────
with gr.Blocks(title="Log Analysis Agent") as demo:

    gr.Markdown(
        "# 🔍 Log Analysis Agent (Claude Opus 4.6)\n"
        "Upload your log file, then ask questions in plain English. "
        "Claude will choose the right tools, query the data, and explain what it finds.\n\n"
        "> **No raw SQL or Python needed** — the model only reports what the tools return."
    )

    # ── 1) Upload ─────────────────────────────────────────────────────────────
    with gr.Accordion("1) Upload logs", open=True):
        with gr.Row():
            file_in     = gr.File(label="Upload logs (.csv / .jsonl / .xlsx)")
            profile_btn = gr.Button("📊 Show dataset info")
        preview = gr.Dataframe(label="Preview (first 15 rows)", interactive=False)
        info    = gr.Markdown()
        file_in.upload(load_logs_gradio, inputs=file_in, outputs=preview)
        profile_btn.click(dataset_profile, outputs=info)

    # ── 2) Chat ───────────────────────────────────────────────────────────────
    gr.Markdown("## 2) Ask the agent")

    chatbot = gr.Chatbot(label="Conversation", height=520)

    with gr.Row():
        msg = gr.Textbox(
            placeholder=(
                "e.g. 'What are the most common errors?' "
                "or 'Show errors from payment-service last week'"
            ),
            label="Your question",
            scale=5,
        )
        send_btn = gr.Button("Send", variant="primary", scale=1)

    clear_btn = gr.Button("🗑️ Clear chat")

    gr.Examples(
        examples=[
            ["What data have I uploaded?"],
            ["What are the top 5 error types?"],
            ["Which service has the most errors?"],
            ["Show errors between 2024-01-01 and 2024-01-31"],
            ["Summarise the biggest issues and suggest next steps"],
        ],
        inputs=msg,
    )

    # ── Events — streaming requires every=True ────────────────────────────────
    send_btn.click(
        run_agent,
        inputs=[msg, chatbot],
        outputs=[msg, chatbot],
    )
    msg.submit(
        run_agent,
        inputs=[msg, chatbot],
        outputs=[msg, chatbot],
    )
    clear_btn.click(lambda: ([], ""), outputs=[chatbot, msg])

demo.launch(share=True)


/tmp/ipykernel_1794/4025342502.py:496: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Conversation", height=520)
/tmp/ipykernel_1794/4025342502.py:496: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Conversation", height=520)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ad6a19cd6cb497abde.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
